Create groww_signup_base
It gives us a common starting point for all five product datasets.

In [0]:
%sql
CREATE OR REPLACE TABLE groww_signup_base AS
SELECT
    third_party_id,
    signup_ts,
    signup_platform,
    kyc_at_signup,
    kra_flag,
    age,
    income,
    occupation,
    kyc_city,
    kyc_state,
    kyc_tier,
    utm_source,
    utm_medium,
    utm_campaign,
    acquisition_channel,
    acquisition_subchannel,

    first_esign_ts,
    aof_ts,
    stock_onboarding_ts,
    first_bank_added_ts,
    selfie_verified_ts,

    fid_ts,
    fid_product,

    -- User's signup-based observation windows
    signup_ts + INTERVAL 7 DAYS AS day7_cutoff,
    signup_ts + INTERVAL 90 DAYS AS day90_cutoff

FROM users
WHERE third_party_id IS NOT NULL
  AND signup_ts IS NOT NULL;

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql

CREATE OR REPLACE TABLE default.groww_day7_features AS

WITH base AS (
    SELECT
        third_party_id,
        signup_ts,
        day7_cutoff,

        signup_platform,
        acquisition_channel,
        acquisition_subchannel,
        utm_source,
        utm_medium,
        utm_campaign,

        kyc_at_signup,
        kra_flag,
        age,
        income,
        occupation,
        kyc_tier,

        first_esign_ts,
        aof_ts,
        stock_onboarding_ts,
        first_bank_added_ts,
        selfie_verified_ts,
        fid_ts,
        fid_product

    FROM default.groww_signup_base
),

/* --------------------------------------------------
   1. STOCKS
   Attempts use created_at.
   Success uses SUCCESS + exchange_update_time.
-------------------------------------------------- */

stocks AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day7_cutoff
            THEN s.groww_order_id
        END) AS stocks_attempted_count_7d,

        COUNT(DISTINCT CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day7_cutoff
             AND s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day7_cutoff
            THEN s.groww_order_id
        END) AS stocks_successful_count_7d,

        SUM(CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day7_cutoff
             AND s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day7_cutoff
            THEN COALESCE(s.original_brokerage, 0)
            ELSE 0
        END) AS stocks_revenue_7d,

        SUM(CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day7_cutoff
             AND s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day7_cutoff
            THEN COALESCE(s.turnover_inr, 0)
            ELSE 0
        END) AS stocks_turnover_7d,

        SUM(CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day7_cutoff
             AND s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day7_cutoff
             AND s.buy_sell = 'B'
            THEN COALESCE(s.turnover_inr, 0)
            ELSE 0
        END) AS stocks_buy_turnover_7d,

        SUM(CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day7_cutoff
             AND s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day7_cutoff
             AND s.buy_sell = 'S'
            THEN COALESCE(s.turnover_inr, 0)
            ELSE 0
        END) AS stocks_sell_turnover_7d,

        COUNT(DISTINCT CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day7_cutoff
            THEN TO_DATE(s.created_at)
        END) AS stocks_active_days_7d

    FROM base b
    LEFT JOIN default.stocks_orders s
        ON b.third_party_id = s.third_party_id
    GROUP BY b.third_party_id
),

/* --------------------------------------------------
   2. F&O
   Attempts use order_created_at.
   Execution uses EXECUTED + exchange_update_time.
-------------------------------------------------- */

fno AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN f.order_created_at >= b.signup_ts
             AND f.order_created_at < b.day7_cutoff
            THEN f.groww_order_id
        END) AS fno_attempted_count_7d,

        COUNT(DISTINCT CASE
            WHEN f.order_created_at >= b.signup_ts
             AND f.order_created_at < b.day7_cutoff
             AND f.order_status = 'EXECUTED'
             AND f.exchange_update_time >= b.signup_ts
             AND f.exchange_update_time < b.day7_cutoff
            THEN f.groww_order_id
        END) AS fno_executed_count_7d,

        20.0 * COUNT(DISTINCT CASE
            WHEN f.order_created_at >= b.signup_ts
             AND f.order_created_at < b.day7_cutoff
             AND f.order_status = 'EXECUTED'
             AND f.exchange_update_time >= b.signup_ts
             AND f.exchange_update_time < b.day7_cutoff
            THEN f.groww_order_id
        END) AS fno_revenue_7d,

        COUNT(DISTINCT CASE
            WHEN f.order_created_at >= b.signup_ts
             AND f.order_created_at < b.day7_cutoff
             AND f.order_status = 'EXECUTED'
             AND f.exchange_update_time >= b.signup_ts
             AND f.exchange_update_time < b.day7_cutoff
             AND f.contract_type = 'FUT'
            THEN f.groww_order_id
        END) AS fno_futures_orders_7d,

        COUNT(DISTINCT CASE
            WHEN f.order_created_at >= b.signup_ts
             AND f.order_created_at < b.day7_cutoff
             AND f.order_status = 'EXECUTED'
             AND f.exchange_update_time >= b.signup_ts
             AND f.exchange_update_time < b.day7_cutoff
             AND f.contract_type = 'OPT'
            THEN f.groww_order_id
        END) AS fno_options_orders_7d,

        COUNT(DISTINCT CASE
            WHEN f.order_created_at >= b.signup_ts
             AND f.order_created_at < b.day7_cutoff
            THEN TO_DATE(f.order_created_at)
        END) AS fno_active_days_7d

    FROM base b
    LEFT JOIN default.fno_orders f
        ON b.third_party_id = f.third_party_id
    GROUP BY b.third_party_id
),

/* --------------------------------------------------
   3. MUTUAL FUNDS
   Attempts use created_at.
   Completed transactions use order_completed_at.
   Purchases and redemptions are separated.
-------------------------------------------------- */

mf AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day7_cutoff
            THEN m.groww_order_id
        END) AS mf_attempted_count_7d,

        COUNT(DISTINCT CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day7_cutoff
             AND m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day7_cutoff
            THEN m.groww_order_id
        END) AS mf_completed_count_7d,

        2.0 * COUNT(DISTINCT CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day7_cutoff
             AND m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day7_cutoff
            THEN m.groww_order_id
        END) AS mf_revenue_7d,

        SUM(CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day7_cutoff
             AND m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day7_cutoff
             AND m.buy_sell = 'P'
            THEN COALESCE(m.order_val, 0)
            ELSE 0
        END) AS mf_purchase_value_7d,

        SUM(CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day7_cutoff
             AND m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day7_cutoff
             AND m.buy_sell = 'R'
            THEN COALESCE(m.order_val, 0)
            ELSE 0
        END) AS mf_redemption_value_7d,

        COUNT(DISTINCT CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day7_cutoff
             AND m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day7_cutoff
             AND m.order_kind = 'SIP'
            THEN m.groww_order_id
        END) AS mf_sip_orders_7d,

        COUNT(DISTINCT CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day7_cutoff
            THEN TO_DATE(m.created_at)
        END) AS mf_active_days_7d

    FROM base b
    LEFT JOIN default.mf_orders m
        ON b.third_party_id = m.third_party_id
    GROUP BY b.third_party_id
),

/* --------------------------------------------------
   4. IPO
   Attempts use created_at.
   Success requires the business-defined payment rule
   and payment_success_time inside the observation window.
-------------------------------------------------- */

ipo AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN i.created_at >= b.signup_ts
             AND i.created_at < b.day7_cutoff
            THEN i.groww_order_id
        END) AS ipo_attempted_count_7d,

        COUNT(DISTINCT CASE
            WHEN i.created_at >= b.signup_ts
             AND i.created_at < b.day7_cutoff
             AND i.order_status IN (
                 'ALLOTMENT_DONE',
                 'NOT_ALLOTED',
                 'APPROVED'
             )
             AND i.new_payment_success_flag = TRUE
             AND i.payment_success_time >= b.signup_ts
             AND i.payment_success_time < b.day7_cutoff
            THEN i.groww_order_id
        END) AS ipo_payment_success_count_7d,

        COUNT(DISTINCT CASE
            WHEN i.created_at >= b.signup_ts
             AND i.created_at < b.day7_cutoff
            THEN TO_DATE(i.created_at)
        END) AS ipo_active_days_7d

    FROM base b
    LEFT JOIN default.ipo_orders i
        ON b.third_party_id = i.third_party_id
    GROUP BY b.third_party_id
),

/* --------------------------------------------------
   5. COMMODITIES
   Only order attempts are measured within day 7.
   No execution/revenue claim is made.
-------------------------------------------------- */

commodity AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN c.created_at >= b.signup_ts
             AND c.created_at < b.day7_cutoff
            THEN c.groww_order_id
        END) AS commodity_order_attempts_7d,

        COUNT(DISTINCT CASE
            WHEN c.created_at >= b.signup_ts
             AND c.created_at < b.day7_cutoff
            THEN TO_DATE(c.created_at)
        END) AS commodity_active_days_7d,

        COUNT(DISTINCT CASE
            WHEN c.created_at >= b.signup_ts
             AND c.created_at < b.day7_cutoff
            THEN c.underlying
        END) AS commodity_underlyings_7d

    FROM base b
    LEFT JOIN default.commodity_orders c
        ON b.third_party_id = c.third_party_id
    GROUP BY b.third_party_id
),

/* --------------------------------------------------
   6. FIRST PRODUCT ACTIVITY
   Based on order placement timestamps.
-------------------------------------------------- */

activity_events AS (
    SELECT
        b.third_party_id,
        s.created_at AS activity_ts
    FROM base b
    JOIN default.stocks_orders s
      ON b.third_party_id = s.third_party_id
    WHERE s.created_at >= b.signup_ts
      AND s.created_at < b.day7_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        f.order_created_at
    FROM base b
    JOIN default.fno_orders f
      ON b.third_party_id = f.third_party_id
    WHERE f.order_created_at >= b.signup_ts
      AND f.order_created_at < b.day7_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        m.created_at
    FROM base b
    JOIN default.mf_orders m
      ON b.third_party_id = m.third_party_id
    WHERE m.created_at >= b.signup_ts
      AND m.created_at < b.day7_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        i.created_at
    FROM base b
    JOIN default.ipo_orders i
      ON b.third_party_id = i.third_party_id
    WHERE i.created_at >= b.signup_ts
      AND i.created_at < b.day7_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        c.created_at
    FROM base b
    JOIN default.commodity_orders c
      ON b.third_party_id = c.third_party_id
    WHERE c.created_at >= b.signup_ts
      AND c.created_at < b.day7_cutoff
),

first_activity AS (
    SELECT
        third_party_id,
        MIN(activity_ts) AS first_activity_ts
    FROM activity_events
    GROUP BY third_party_id
),

/* --------------------------------------------------
   7. ACTIVE DAYS ACROSS ALL PRODUCTS
-------------------------------------------------- */

activity_dates AS (
    SELECT third_party_id, TO_DATE(activity_ts) AS activity_date
    FROM activity_events
),

overall_activity AS (
    SELECT
        third_party_id,
        COUNT(DISTINCT activity_date) AS active_days_7d
    FROM activity_dates
    GROUP BY third_party_id
)

SELECT
    b.third_party_id,
    b.signup_ts,
    b.signup_platform,

    b.acquisition_channel,
    b.acquisition_subchannel,
    b.utm_source,
    b.utm_medium,
    b.utm_campaign,

    b.kyc_at_signup,
    b.kra_flag,
    b.age,
    b.income,
    b.occupation,
    b.kyc_tier,

    /* Onboarding milestones */

    CASE WHEN b.first_esign_ts >= b.signup_ts
          AND b.first_esign_ts < b.day7_cutoff
         THEN 1 ELSE 0 END AS esign_within_7d,

    CASE WHEN b.aof_ts >= b.signup_ts
          AND b.aof_ts < b.day7_cutoff
         THEN 1 ELSE 0 END AS aof_within_7d,

    CASE WHEN b.stock_onboarding_ts >= b.signup_ts
          AND b.stock_onboarding_ts < b.day7_cutoff
         THEN 1 ELSE 0 END AS stock_onboarding_within_7d,

    CASE WHEN b.first_bank_added_ts >= b.signup_ts
          AND b.first_bank_added_ts < b.day7_cutoff
         THEN 1 ELSE 0 END AS bank_added_within_7d,

    CASE WHEN b.selfie_verified_ts >= b.signup_ts
          AND b.selfie_verified_ts < b.day7_cutoff
         THEN 1 ELSE 0 END AS selfie_verified_within_7d,

    CASE WHEN b.fid_ts >= b.signup_ts
          AND b.fid_ts < b.day7_cutoff
         THEN 1 ELSE 0 END AS first_investment_within_7d,

    CASE WHEN b.fid_ts >= b.signup_ts
          AND b.fid_ts < b.day7_cutoff
         THEN b.fid_product
         ELSE NULL
    END AS first_investment_product_7d,

    (
        CASE WHEN b.first_esign_ts >= b.signup_ts
                   AND b.first_esign_ts < b.day7_cutoff THEN 1 ELSE 0 END
      + CASE WHEN b.aof_ts >= b.signup_ts
                   AND b.aof_ts < b.day7_cutoff THEN 1 ELSE 0 END
      + CASE WHEN b.stock_onboarding_ts >= b.signup_ts
                   AND b.stock_onboarding_ts < b.day7_cutoff THEN 1 ELSE 0 END
      + CASE WHEN b.first_bank_added_ts >= b.signup_ts
                   AND b.first_bank_added_ts < b.day7_cutoff THEN 1 ELSE 0 END
      + CASE WHEN b.selfie_verified_ts >= b.signup_ts
                   AND b.selfie_verified_ts < b.day7_cutoff THEN 1 ELSE 0 END
      + CASE WHEN b.fid_ts >= b.signup_ts
                   AND b.fid_ts < b.day7_cutoff THEN 1 ELSE 0 END
    ) AS onboarding_milestones_completed_7d,

    (
        (
            CASE WHEN b.first_esign_ts >= b.signup_ts
                       AND b.first_esign_ts < b.day7_cutoff THEN 1 ELSE 0 END
          + CASE WHEN b.aof_ts >= b.signup_ts
                       AND b.aof_ts < b.day7_cutoff THEN 1 ELSE 0 END
          + CASE WHEN b.stock_onboarding_ts >= b.signup_ts
                       AND b.stock_onboarding_ts < b.day7_cutoff THEN 1 ELSE 0 END
          + CASE WHEN b.first_bank_added_ts >= b.signup_ts
                       AND b.first_bank_added_ts < b.day7_cutoff THEN 1 ELSE 0 END
          + CASE WHEN b.selfie_verified_ts >= b.signup_ts
                       AND b.selfie_verified_ts < b.day7_cutoff THEN 1 ELSE 0 END
          + CASE WHEN b.fid_ts >= b.signup_ts
                       AND b.fid_ts < b.day7_cutoff THEN 1 ELSE 0 END
        ) / 6.0
    ) AS onboarding_completion_rate_7d,

    /* Stocks */

    COALESCE(s.stocks_attempted_count_7d, 0) AS stocks_attempted_count_7d,
    COALESCE(s.stocks_successful_count_7d, 0) AS stocks_successful_count_7d,
    CASE WHEN COALESCE(s.stocks_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END AS stocks_attempted_flag_7d,
    CASE WHEN COALESCE(s.stocks_successful_count_7d, 0) > 0 THEN 1 ELSE 0 END AS stocks_successful_flag_7d,
    COALESCE(s.stocks_revenue_7d, 0) AS stocks_revenue_7d,
    COALESCE(s.stocks_turnover_7d, 0) AS stocks_turnover_7d,
    COALESCE(s.stocks_buy_turnover_7d, 0) AS stocks_buy_turnover_7d,
    COALESCE(s.stocks_sell_turnover_7d, 0) AS stocks_sell_turnover_7d,
    COALESCE(s.stocks_active_days_7d, 0) AS stocks_active_days_7d,

    /* F&O */

    COALESCE(f.fno_attempted_count_7d, 0) AS fno_attempted_count_7d,
    COALESCE(f.fno_executed_count_7d, 0) AS fno_executed_count_7d,
    CASE WHEN COALESCE(f.fno_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END AS fno_attempted_flag_7d,
    CASE WHEN COALESCE(f.fno_executed_count_7d, 0) > 0 THEN 1 ELSE 0 END AS fno_executed_flag_7d,
    COALESCE(f.fno_revenue_7d, 0) AS fno_revenue_7d,
    COALESCE(f.fno_futures_orders_7d, 0) AS fno_futures_orders_7d,
    COALESCE(f.fno_options_orders_7d, 0) AS fno_options_orders_7d,
    COALESCE(f.fno_active_days_7d, 0) AS fno_active_days_7d,

    /* Mutual funds */

    COALESCE(m.mf_attempted_count_7d, 0) AS mf_attempted_count_7d,
    COALESCE(m.mf_completed_count_7d, 0) AS mf_completed_count_7d,
    CASE WHEN COALESCE(m.mf_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END AS mf_attempted_flag_7d,
    CASE WHEN COALESCE(m.mf_completed_count_7d, 0) > 0 THEN 1 ELSE 0 END AS mf_completed_flag_7d,
    COALESCE(m.mf_revenue_7d, 0) AS mf_revenue_7d,
    COALESCE(m.mf_purchase_value_7d, 0) AS mf_purchase_value_7d,
    COALESCE(m.mf_redemption_value_7d, 0) AS mf_redemption_value_7d,
    COALESCE(m.mf_sip_orders_7d, 0) AS mf_sip_orders_7d,
    COALESCE(m.mf_active_days_7d, 0) AS mf_active_days_7d,

    /* IPO */

    COALESCE(i.ipo_attempted_count_7d, 0) AS ipo_attempted_count_7d,
    COALESCE(i.ipo_payment_success_count_7d, 0) AS ipo_payment_success_count_7d,
    CASE WHEN COALESCE(i.ipo_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END AS ipo_attempted_flag_7d,
    CASE WHEN COALESCE(i.ipo_payment_success_count_7d, 0) > 0 THEN 1 ELSE 0 END AS ipo_payment_success_flag_7d,
    COALESCE(i.ipo_payment_success_count_7d, 0) * 1.0 AS ipo_revenue_7d,
    COALESCE(i.ipo_active_days_7d, 0) AS ipo_active_days_7d,

    /* Commodities: attempts only */

    COALESCE(c.commodity_order_attempts_7d, 0) AS commodity_order_attempts_7d,
    CASE WHEN COALESCE(c.commodity_order_attempts_7d, 0) > 0 THEN 1 ELSE 0 END AS commodity_attempted_flag_7d,
    COALESCE(c.commodity_active_days_7d, 0) AS commodity_active_days_7d,
    COALESCE(c.commodity_underlyings_7d, 0) AS commodity_underlyings_7d,

    /* Overall engagement */

    COALESCE(s.stocks_attempted_count_7d, 0)
      + COALESCE(f.fno_attempted_count_7d, 0)
      + COALESCE(m.mf_attempted_count_7d, 0)
      + COALESCE(i.ipo_attempted_count_7d, 0)
      + COALESCE(c.commodity_order_attempts_7d, 0)
        AS total_attempted_orders_7d,

    COALESCE(s.stocks_successful_count_7d, 0)
      + COALESCE(f.fno_executed_count_7d, 0)
      + COALESCE(m.mf_completed_count_7d, 0)
      + COALESCE(i.ipo_payment_success_count_7d, 0)
        AS successful_transactions_7d,

    COALESCE(s.stocks_revenue_7d, 0)
      + COALESCE(f.fno_revenue_7d, 0)
      + COALESCE(m.mf_revenue_7d, 0)
      + COALESCE(i.ipo_payment_success_count_7d, 0)
        AS observable_revenue_7d,

    COALESCE(s.stocks_attempted_count_7d, 0)
      + COALESCE(f.fno_attempted_count_7d, 0)
      + COALESCE(m.mf_attempted_count_7d, 0)
      + COALESCE(i.ipo_attempted_count_7d, 0)
        AS timestamped_product_attempts_7d,

    CASE WHEN COALESCE(s.stocks_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(f.fno_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(m.mf_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(i.ipo_attempted_count_7d, 0) > 0 THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(c.commodity_order_attempts_7d, 0) > 0 THEN 1 ELSE 0 END
        AS products_attempted_7d,

    CASE WHEN COALESCE(s.stocks_successful_count_7d, 0) > 0 THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(f.fno_executed_count_7d, 0) > 0 THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(m.mf_completed_count_7d, 0) > 0 THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(i.ipo_payment_success_count_7d, 0) > 0 THEN 1 ELSE 0 END
        AS products_successful_7d,

    CASE
        WHEN (
            COALESCE(s.stocks_attempted_count_7d, 0)
          + COALESCE(f.fno_attempted_count_7d, 0)
          + COALESCE(m.mf_attempted_count_7d, 0)
          + COALESCE(i.ipo_attempted_count_7d, 0)
        ) > 0
        THEN (
            COALESCE(s.stocks_successful_count_7d, 0)
          + COALESCE(f.fno_executed_count_7d, 0)
          + COALESCE(m.mf_completed_count_7d, 0)
          + COALESCE(i.ipo_payment_success_count_7d, 0)
        ) * 1.0 / (
            COALESCE(s.stocks_attempted_count_7d, 0)
          + COALESCE(f.fno_attempted_count_7d, 0)
          + COALESCE(m.mf_attempted_count_7d, 0)
          + COALESCE(i.ipo_attempted_count_7d, 0)
        )
        ELSE 0
    END AS observable_success_rate_7d,

    COALESCE(a.active_days_7d, 0) AS active_days_7d,

    CASE
        WHEN COALESCE(a.active_days_7d, 0) > 0
        THEN (
            COALESCE(s.stocks_attempted_count_7d, 0)
          + COALESCE(f.fno_attempted_count_7d, 0)
          + COALESCE(m.mf_attempted_count_7d, 0)
          + COALESCE(i.ipo_attempted_count_7d, 0)
          + COALESCE(c.commodity_order_attempts_7d, 0)
        ) * 1.0 / a.active_days_7d
        ELSE 0
    END AS orders_per_active_day_7d,

    fa.first_activity_ts,

    CASE WHEN fa.first_activity_ts IS NOT NULL
         THEN DATEDIFF(TO_DATE(fa.first_activity_ts), TO_DATE(b.signup_ts))
         ELSE NULL
    END AS days_to_first_activity,

    CASE WHEN b.fid_ts >= b.signup_ts
           AND b.fid_ts < b.day7_cutoff
         THEN DATEDIFF(TO_DATE(b.fid_ts), TO_DATE(b.signup_ts))
         ELSE NULL
    END AS days_to_first_investment

FROM base b

LEFT JOIN stocks s
    ON b.third_party_id = s.third_party_id

LEFT JOIN fno f
    ON b.third_party_id = f.third_party_id

LEFT JOIN mf m
    ON b.third_party_id = m.third_party_id

LEFT JOIN ipo i
    ON b.third_party_id = i.third_party_id

LEFT JOIN commodity c
    ON b.third_party_id = c.third_party_id

LEFT JOIN overall_activity a
    ON b.third_party_id = a.third_party_id

LEFT JOIN first_activity fa
    ON b.third_party_id = fa.third_party_id;


In [0]:
%sql
DESCRIBE TABLE default.groww_day7_features;

Little correction in groww-7_feature table

In [0]:
%sql

CREATE OR REPLACE TABLE default.groww_day7_features AS

SELECT
    * EXCEPT (
        onboarding_milestones_completed_7d,
        onboarding_completion_rate_7d
    ),

    (
        esign_within_7d
        + aof_within_7d
        + stock_onboarding_within_7d
        + bank_added_within_7d
        + selfie_verified_within_7d
    ) AS onboarding_steps_completed_7d,

    (
        esign_within_7d
        + aof_within_7d
        + stock_onboarding_within_7d
        + bank_added_within_7d
        + selfie_verified_within_7d
    ) / 5.0 AS onboarding_completion_rate_7d

FROM default.groww_day7_features;


In [0]:
%sql
SELECT *
FROM default.groww_day7_features
WHERE total_attempted_orders_7d > 0
ORDER BY observable_revenue_7d DESC
LIMIT 20;

Stage 2: building groww_day90_outcomes, which will measure what each user actually generated during their first 90 days.

In [0]:
%sql

CREATE OR REPLACE TABLE default.groww_day90_outcomes AS

WITH base AS (
    SELECT
        third_party_id,
        signup_ts,
        day90_cutoff
    FROM default.groww_signup_base
    WHERE day90_cutoff <= CURRENT_TIMESTAMP()
),

/* ==================================================
   1. STOCKS
   Revenue = original_brokerage for SUCCESS orders
================================================== */

stocks AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN s.created_at >= b.signup_ts
             AND s.created_at < b.day90_cutoff
            THEN s.groww_order_id
        END) AS stocks_attempted_90d,

        COUNT(DISTINCT CASE
            WHEN s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day90_cutoff
            THEN s.groww_order_id
        END) AS stocks_successful_90d,

        SUM(CASE
            WHEN s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day90_cutoff
            THEN COALESCE(s.original_brokerage, 0)
            ELSE 0
        END) AS stocks_revenue_90d,

        SUM(CASE
            WHEN s.order_status = 'SUCCESS'
             AND s.exchange_update_time >= b.signup_ts
             AND s.exchange_update_time < b.day90_cutoff
            THEN COALESCE(s.turnover_inr, 0)
            ELSE 0
        END) AS stocks_turnover_90d

    FROM base b
    LEFT JOIN default.stocks_orders s
        ON b.third_party_id = s.third_party_id
    GROUP BY b.third_party_id
),

/* ==================================================
   2. F&O
   Revenue = ₹20 per distinct EXECUTED order
================================================== */

fno AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN f.order_created_at >= b.signup_ts
             AND f.order_created_at < b.day90_cutoff
            THEN f.groww_order_id
        END) AS fno_attempted_90d,

        COUNT(DISTINCT CASE
            WHEN f.order_status = 'EXECUTED'
             AND f.exchange_update_time >= b.signup_ts
             AND f.exchange_update_time < b.day90_cutoff
            THEN f.groww_order_id
        END) AS fno_executed_90d,

        20.0 * COUNT(DISTINCT CASE
            WHEN f.order_status = 'EXECUTED'
             AND f.exchange_update_time >= b.signup_ts
             AND f.exchange_update_time < b.day90_cutoff
            THEN f.groww_order_id
        END) AS fno_revenue_90d

    FROM base b
    LEFT JOIN default.fno_orders f
        ON b.third_party_id = f.third_party_id
    GROUP BY b.third_party_id
),

/* ==================================================
   3. MUTUAL FUNDS
   Revenue = ₹2 per distinct COMPLETED order
   Purchases and redemptions are separate
================================================== */

mf AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN m.created_at >= b.signup_ts
             AND m.created_at < b.day90_cutoff
            THEN m.groww_order_id
        END) AS mf_attempted_90d,

        COUNT(DISTINCT CASE
            WHEN m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day90_cutoff
            THEN m.groww_order_id
        END) AS mf_completed_90d,

        2.0 * COUNT(DISTINCT CASE
            WHEN m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day90_cutoff
            THEN m.groww_order_id
        END) AS mf_revenue_90d,

        SUM(CASE
            WHEN m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day90_cutoff
             AND m.buy_sell = 'P'
            THEN COALESCE(m.order_val, 0)
            ELSE 0
        END) AS mf_purchase_value_90d,

        SUM(CASE
            WHEN m.order_status = 'COMPLETED'
             AND m.order_completed_at >= b.signup_ts
             AND m.order_completed_at < b.day90_cutoff
             AND m.buy_sell = 'R'
            THEN COALESCE(m.order_val, 0)
            ELSE 0
        END) AS mf_redemption_value_90d

    FROM base b
    LEFT JOIN default.mf_orders m
        ON b.third_party_id = m.third_party_id
    GROUP BY b.third_party_id
),

/* ==================================================
   4. IPO
   Revenue = ₹1 per successful application
================================================== */

ipo AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN i.created_at >= b.signup_ts
             AND i.created_at < b.day90_cutoff
            THEN i.groww_order_id
        END) AS ipo_attempted_90d,

        COUNT(DISTINCT CASE
            WHEN i.order_status IN (
                'ALLOTMENT_DONE',
                'NOT_ALLOTED',
                'APPROVED'
            )
             AND i.new_payment_success_flag = TRUE
             AND i.payment_success_time >= b.signup_ts
             AND i.payment_success_time < b.day90_cutoff
            THEN i.groww_order_id
        END) AS ipo_successful_90d

    FROM base b
    LEFT JOIN default.ipo_orders i
        ON b.third_party_id = i.third_party_id
    GROUP BY b.third_party_id
),

/* ==================================================
   5. COMMODITIES
   Sensitivity proxy only:
   final EXECUTED status + order creation within 90d
   NOT a confirmed execution-time measurement
================================================== */

commodity AS (
    SELECT
        b.third_party_id,

        COUNT(DISTINCT CASE
            WHEN c.created_at >= b.signup_ts
             AND c.created_at < b.day90_cutoff
            THEN c.groww_order_id
        END) AS commodity_attempts_90d,

        COUNT(DISTINCT CASE
            WHEN c.created_at >= b.signup_ts
             AND c.created_at < b.day90_cutoff
             AND c.order_status = 'EXECUTED'
            THEN c.groww_order_id
        END) AS commodity_executed_proxy_90d,

        20.0 * COUNT(DISTINCT CASE
            WHEN c.created_at >= b.signup_ts
             AND c.created_at < b.day90_cutoff
             AND c.order_status = 'EXECUTED'
            THEN c.groww_order_id
        END) AS commodity_revenue_proxy_90d

    FROM base b
    LEFT JOIN default.commodity_orders c
        ON b.third_party_id = c.third_party_id
    GROUP BY b.third_party_id
),

/* ==================================================
   6. ACTIVE DAYS
   Distinct dates on which users placed orders
================================================== */

activity_events AS (

    SELECT
        b.third_party_id,
        TO_DATE(s.created_at) AS activity_date
    FROM base b
    JOIN default.stocks_orders s
        ON b.third_party_id = s.third_party_id
    WHERE s.created_at >= b.signup_ts
      AND s.created_at < b.day90_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        TO_DATE(f.order_created_at)
    FROM base b
    JOIN default.fno_orders f
        ON b.third_party_id = f.third_party_id
    WHERE f.order_created_at >= b.signup_ts
      AND f.order_created_at < b.day90_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        TO_DATE(m.created_at)
    FROM base b
    JOIN default.mf_orders m
        ON b.third_party_id = m.third_party_id
    WHERE m.created_at >= b.signup_ts
      AND m.created_at < b.day90_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        TO_DATE(i.created_at)
    FROM base b
    JOIN default.ipo_orders i
        ON b.third_party_id = i.third_party_id
    WHERE i.created_at >= b.signup_ts
      AND i.created_at < b.day90_cutoff

    UNION ALL

    SELECT
        b.third_party_id,
        TO_DATE(c.created_at)
    FROM base b
    JOIN default.commodity_orders c
        ON b.third_party_id = c.third_party_id
    WHERE c.created_at >= b.signup_ts
      AND c.created_at < b.day90_cutoff
),

activity AS (
    SELECT
        third_party_id,
        COUNT(DISTINCT activity_date) AS active_days_90d
    FROM activity_events
    GROUP BY third_party_id
),

/* ==================================================
   7. COMBINE PRODUCT OUTCOMES
================================================== */

combined AS (
    SELECT
        b.third_party_id,
        b.signup_ts,
        b.day90_cutoff,

        /* Stocks */
        COALESCE(s.stocks_attempted_90d, 0)
            AS stocks_attempted_90d,
        COALESCE(s.stocks_successful_90d, 0)
            AS stocks_successful_90d,
        COALESCE(s.stocks_revenue_90d, 0)
            AS stocks_revenue_90d,
        COALESCE(s.stocks_turnover_90d, 0)
            AS stocks_turnover_90d,

        /* F&O */
        COALESCE(f.fno_attempted_90d, 0)
            AS fno_attempted_90d,
        COALESCE(f.fno_executed_90d, 0)
            AS fno_executed_90d,
        COALESCE(f.fno_revenue_90d, 0)
            AS fno_revenue_90d,

        /* Mutual funds */
        COALESCE(m.mf_attempted_90d, 0)
            AS mf_attempted_90d,
        COALESCE(m.mf_completed_90d, 0)
            AS mf_completed_90d,
        COALESCE(m.mf_revenue_90d, 0)
            AS mf_revenue_90d,
        COALESCE(m.mf_purchase_value_90d, 0)
            AS mf_purchase_value_90d,
        COALESCE(m.mf_redemption_value_90d, 0)
            AS mf_redemption_value_90d,

        /* IPO */
        COALESCE(i.ipo_attempted_90d, 0)
            AS ipo_attempted_90d,
        COALESCE(i.ipo_successful_90d, 0)
            AS ipo_successful_90d,

        /* Commodities */
        COALESCE(c.commodity_attempts_90d, 0)
            AS commodity_attempts_90d,
        COALESCE(c.commodity_executed_proxy_90d, 0)
            AS commodity_executed_proxy_90d,
        COALESCE(c.commodity_revenue_proxy_90d, 0)
            AS commodity_revenue_proxy_90d,

        /* Engagement */
        COALESCE(a.active_days_90d, 0)
            AS active_days_90d,

        /* Primary observable revenue */
        COALESCE(s.stocks_revenue_90d, 0)
        + COALESCE(f.fno_revenue_90d, 0)
        + COALESCE(m.mf_revenue_90d, 0)
        + COALESCE(i.ipo_successful_90d, 0) * 1.0
            AS observable_revenue_90d,

        /* Commodity-inclusive sensitivity revenue */
        COALESCE(s.stocks_revenue_90d, 0)
        + COALESCE(f.fno_revenue_90d, 0)
        + COALESCE(m.mf_revenue_90d, 0)
        + COALESCE(i.ipo_successful_90d, 0) * 1.0
        + COALESCE(c.commodity_revenue_proxy_90d, 0)
            AS commodity_inclusive_revenue_proxy_90d,

        /* Successful transactions */
        COALESCE(s.stocks_successful_90d, 0)
        + COALESCE(f.fno_executed_90d, 0)
        + COALESCE(m.mf_completed_90d, 0)
        + COALESCE(i.ipo_successful_90d, 0)
            AS observable_successful_transactions_90d,

        /* Number of successful products */
        CASE WHEN COALESCE(s.stocks_successful_90d, 0) > 0
             THEN 1 ELSE 0 END
        + CASE WHEN COALESCE(f.fno_executed_90d, 0) > 0
             THEN 1 ELSE 0 END
        + CASE WHEN COALESCE(m.mf_completed_90d, 0) > 0
             THEN 1 ELSE 0 END
        + CASE WHEN COALESCE(i.ipo_successful_90d, 0) > 0
             THEN 1 ELSE 0 END
            AS products_successful_90d

    FROM base b

    LEFT JOIN stocks s
        ON b.third_party_id = s.third_party_id

    LEFT JOIN fno f
        ON b.third_party_id = f.third_party_id

    LEFT JOIN mf m
        ON b.third_party_id = m.third_party_id

    LEFT JOIN ipo i
        ON b.third_party_id = i.third_party_id

    LEFT JOIN commodity c
        ON b.third_party_id = c.third_party_id

    LEFT JOIN activity a
        ON b.third_party_id = a.third_party_id
),

/* ==================================================
   8. PRIMARY HIGH-VALUE RANKING
   Only users with positive observable revenue
================================================== */

observable_ranked AS (
    SELECT
        third_party_id,

        NTILE(5) OVER (
            ORDER BY observable_revenue_90d DESC,
                     third_party_id
        ) AS observable_value_quintile

    FROM combined
    WHERE observable_revenue_90d > 0
),

/* ==================================================
   9. COMMODITY SENSITIVITY RANKING
   Independent ranking population
================================================== */

sensitivity_ranked AS (
    SELECT
        third_party_id,

        NTILE(5) OVER (
            ORDER BY commodity_inclusive_revenue_proxy_90d DESC,
                     third_party_id
        ) AS commodity_inclusive_value_quintile

    FROM combined
    WHERE commodity_inclusive_revenue_proxy_90d > 0
)

/* ==================================================
   10. FINAL OUTCOME TABLE
================================================== */

SELECT
    c.*,

    /* Primary high-value label */
    CASE
        WHEN c.observable_revenue_90d > 0
         AND o.observable_value_quintile = 1
        THEN 1
        ELSE 0
    END AS high_value_90d,

    /* Sensitivity high-value label */
    CASE
        WHEN c.commodity_inclusive_revenue_proxy_90d > 0
         AND s.commodity_inclusive_value_quintile = 1
        THEN 1
        ELSE 0
    END AS high_value_90d_commodity_sensitivity,

    o.observable_value_quintile,

    s.commodity_inclusive_value_quintile

FROM combined c

LEFT JOIN observable_ranked o
    ON c.third_party_id = o.third_party_id

LEFT JOIN sensitivity_ranked s
    ON c.third_party_id = s.third_party_id;


Excellent. These results are actually quite informative. Your primary and commodity-inclusive high-value groups are almost identical.
1. Understanding your results:
371
High-value in both models

4
Only primary label

4
Only sensitivity label

13,780
Neither label



Total-high-value users:
Label	Users
Primary	375
Commodity sensitivity	375
Common users	371


What does this tell us?
- Both approaches identify exactly 375 high-value users.
- 371 users are common to both groups.
- Only 4 users change from low-value to high-value.
- Only 4 users change from high-value to low-value.
So commodity revenue changes the membership of just 8 users.

----------------------------------------------------------------------
----------------------------------------------------------------------
create the supervised modelling dataset by combining the day-7 features with the day-90 outcomes.

This dataset will contain only users whose full 90-day observation window has elapsed. That is important: users without a complete outcome window cannot be reliably labelled for training.

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_modeling_dataset AS

SELECT
    d7.*,

    -- Day-90 observation boundary
    d90.day90_cutoff,

    -- Stocks
    d90.stocks_attempted_90d,
    d90.stocks_successful_90d,
    d90.stocks_revenue_90d,
    d90.stocks_turnover_90d,

    -- F&O
    d90.fno_attempted_90d,
    d90.fno_executed_90d,
    d90.fno_revenue_90d,

    -- Mutual Funds
    d90.mf_attempted_90d,
    d90.mf_completed_90d,
    d90.mf_revenue_90d,
    d90.mf_purchase_value_90d,
    d90.mf_redemption_value_90d,

    -- IPO
    d90.ipo_attempted_90d,
    d90.ipo_successful_90d,

    -- Commodities
    d90.commodity_attempts_90d,
    d90.commodity_executed_proxy_90d,
    d90.commodity_revenue_proxy_90d,

    -- Overall day-90 outcomes
    d90.active_days_90d,
    d90.observable_revenue_90d,
    d90.commodity_inclusive_revenue_proxy_90d,
    d90.observable_successful_transactions_90d,
    d90.products_successful_90d,

    -- Prediction labels
    d90.high_value_90d,
    d90.high_value_90d_commodity_sensitivity,

    -- Diagnostic quintiles
    d90.observable_value_quintile,
    d90.commodity_inclusive_value_quintile

FROM default.groww_day7_features d7

INNER JOIN default.groww_day90_outcomes d90
    ON d7.third_party_id = d90.third_party_id;

One modelling decision: We'll build two feature sets:
1. Model A — Behaviour-only: onboarding, engagement and product usage, excluding early revenue.
2. Model B — Behaviour + early revenue: the same information plus first-seven-day revenue.
This comparison will tell us whether early monetisation adds meaningful predictive power beyond user behaviour.

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_a_features AS

SELECT
    -- Identifiers and target
    third_party_id,
    signup_ts,
    high_value_90d AS target,

    -- Acquisition and signup profile
    signup_platform,
    acquisition_channel,
    acquisition_subchannel,
    kyc_at_signup,
    kra_flag,
    age,
    income,
    occupation,
    kyc_tier,

    -- Onboarding
    esign_within_7d,
    aof_within_7d,
    stock_onboarding_within_7d,
    bank_added_within_7d,
    selfie_verified_within_7d,
    onboarding_steps_completed_7d,
    onboarding_completion_rate_7d,

    -- First investment behaviour
    first_investment_within_7d,
    first_investment_product_7d,
    days_to_first_investment,

    -- Stocks
    stocks_attempted_count_7d,
    stocks_successful_count_7d,
    stocks_attempted_flag_7d,
    stocks_successful_flag_7d,
    stocks_active_days_7d,

    -- F&O
    fno_attempted_count_7d,
    fno_executed_count_7d,
    fno_attempted_flag_7d,
    fno_executed_flag_7d,
    fno_futures_orders_7d,
    fno_options_orders_7d,
    fno_active_days_7d,

    -- Mutual Funds
    mf_attempted_count_7d,
    mf_completed_count_7d,
    mf_attempted_flag_7d,
    mf_completed_flag_7d,
    mf_sip_orders_7d,
    mf_active_days_7d,

    -- IPO
    ipo_attempted_count_7d,
    ipo_payment_success_count_7d,
    ipo_attempted_flag_7d,
    ipo_payment_success_flag_7d,
    ipo_active_days_7d,

    -- Commodities: attempts only
    commodity_order_attempts_7d,
    commodity_attempted_flag_7d,
    commodity_active_days_7d,
    commodity_underlyings_7d,

    -- Overall engagement
    total_attempted_orders_7d,
    successful_transactions_7d,
    timestamped_product_attempts_7d,
    products_attempted_7d,
    products_successful_7d,
    observable_success_rate_7d,
    active_days_7d,
    orders_per_active_day_7d,
    days_to_first_activity

FROM default.groww_modeling_dataset;

MODEL B: BEHAVIOUR + EARLY REVENUE
Model B uses exactly the same features, with additional early monetary signals

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_b_features AS

SELECT
    -- Identifiers and target
    third_party_id,
    signup_ts,
    high_value_90d AS target,

    -- Acquisition and signup profile
    signup_platform,
    acquisition_channel,
    acquisition_subchannel,
    kyc_at_signup,
    kra_flag,
    age,
    income,
    occupation,
    kyc_tier,

    -- Onboarding
    esign_within_7d,
    aof_within_7d,
    stock_onboarding_within_7d,
    bank_added_within_7d,
    selfie_verified_within_7d,
    onboarding_steps_completed_7d,
    onboarding_completion_rate_7d,

    -- First investment behaviour
    first_investment_within_7d,
    first_investment_product_7d,
    days_to_first_investment,

    -- Stocks
    stocks_attempted_count_7d,
    stocks_successful_count_7d,
    stocks_attempted_flag_7d,
    stocks_successful_flag_7d,
    stocks_active_days_7d,
    stocks_revenue_7d,
    stocks_turnover_7d,
    stocks_buy_turnover_7d,
    stocks_sell_turnover_7d,

    -- F&O
    fno_attempted_count_7d,
    fno_executed_count_7d,
    fno_attempted_flag_7d,
    fno_executed_flag_7d,
    fno_futures_orders_7d,
    fno_options_orders_7d,
    fno_active_days_7d,
    fno_revenue_7d,

    -- Mutual Funds
    mf_attempted_count_7d,
    mf_completed_count_7d,
    mf_attempted_flag_7d,
    mf_completed_flag_7d,
    mf_sip_orders_7d,
    mf_active_days_7d,
    mf_revenue_7d,
    mf_purchase_value_7d,
    mf_redemption_value_7d,

    -- IPO
    ipo_attempted_count_7d,
    ipo_payment_success_count_7d,
    ipo_attempted_flag_7d,
    ipo_payment_success_flag_7d,
    ipo_active_days_7d,
    ipo_revenue_7d,

    -- Commodities
    commodity_order_attempts_7d,
    commodity_attempted_flag_7d,
    commodity_active_days_7d,
    commodity_underlyings_7d,

    -- Overall engagement
    total_attempted_orders_7d,
    successful_transactions_7d,
    timestamped_product_attempts_7d,
    products_attempted_7d,
    products_successful_7d,
    observable_success_rate_7d,
    active_days_7d,
    orders_per_active_day_7d,
    days_to_first_activity,

    -- Day-90 target
    high_value_90d

FROM default.groww_modeling_dataset;

prepare the chronological train, validation and test split.

In [0]:
%sql
SELECT
    DATE_FORMAT(signup_ts, 'yyyy-MM') AS signup_month,
    COUNT(*) AS total_users,
    SUM(target) AS high_value_users,
    ROUND(100.0 * SUM(target) / COUNT(*), 2)
        AS high_value_percentage
FROM default.groww_model_b_features
GROUP BY DATE_FORMAT(signup_ts, 'yyyy-MM')
ORDER BY signup_month;

Your monthly results show two important things:
-The high-value class is rare: only around 2–3% of all registered users are labelled high-value. That's expected because our target is the top 20% of users with positive observable revenue, not the top 20% of all signups.
- Signup volume is uneven across months, so using whole months would produce a poor 70/15/15 split.
For example, April–June alone account for 8,949 users, while July has 3,554. Instead, we'll split chronologically at exact user-level boundaries, preserving the signup order.
Step 9 — Create chronological split boundaries

-------------------------------------------------------------

-----------------------------------------------------------------
 Create chronological split boundaries
 This creates one shared split table, ensuring Model A and Model B use exactly the same users in each partition.

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_temporal_split AS

WITH ordered_users AS (
    SELECT
        third_party_id,
        signup_ts,

        ROW_NUMBER() OVER (
            ORDER BY signup_ts, third_party_id
        ) AS row_num,

        COUNT(*) OVER () AS total_users

    FROM default.groww_model_b_features
)

SELECT
    third_party_id,
    signup_ts,
    row_num,

    CASE
        WHEN row_num <= FLOOR(total_users * 0.70)
            THEN 'TRAIN'

        WHEN row_num <= FLOOR(total_users * 0.85)
            THEN 'VALIDATION'

        ELSE 'TEST'
    END AS dataset_split

FROM ordered_users;

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_a_split AS

SELECT
    a.*,
    s.dataset_split

FROM default.groww_model_a_features a

INNER JOIN default.groww_temporal_split s
    ON a.third_party_id = s.third_party_id;

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_b_split AS

SELECT
    b.*,
    s.dataset_split

FROM default.groww_model_b_features b

INNER JOIN default.groww_temporal_split s
    ON b.third_party_id = s.third_party_id;

TRAINING : LOGISTIC REGRESSION

In [0]:

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Revenue data is used only for validation metrics, never as a feature.
revenue_pdf = (
    spark.table("default.groww_modeling_dataset")
    .select("third_party_id", "observable_revenue_90d")
    .toPandas()
)

def train_logistic(table_name, model_name):

    source = spark.table(table_name)

    # Identify feature columns using the actual Spark schema.
    excluded = {
        "third_party_id",
        "signup_ts",
        "target",
        "dataset_split"
    }

    feature_cols = [
        c for c in source.columns if c not in excluded
    ]

    numeric_types = (
        "int", "bigint", "double", "float",
        "decimal", "smallint", "tinyint"
    )

    numeric_cols = []
    categorical_cols = []

    for field in source.schema.fields:
        if field.name not in feature_cols:
            continue

        dtype = field.dataType.simpleString()

        if dtype.startswith(numeric_types):
            numeric_cols.append(field.name)
        elif dtype == "string":
            categorical_cols.append(field.name)

    # Bring the relatively small modelling dataset into pandas.
    pdf = source.select(
        "third_party_id",
        "target",
        "dataset_split",
        *feature_cols
    ).toPandas()

    # Ensure numeric fields are numeric.
    for col in numeric_cols:
        pdf[col] = pd.to_numeric(
            pdf[col], errors="coerce"
        )

    # Preprocessing for numeric features.
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Preprocessing for categorical features.
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_cols),
            ("categorical", categorical_transformer, categorical_cols)
        ]
    )

    # Class-weighted baseline for the rare positive class.
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    train = pdf[pdf["dataset_split"] == "TRAIN"].copy()
    val = pdf[pdf["dataset_split"] == "VALIDATION"].copy()

    X_train = train[feature_cols]
    y_train = train["target"].astype(int)

    X_val = val[feature_cols]
    y_val = val["target"].astype(int)

    # Train.
    model.fit(X_train, y_train)

    # Validation probabilities.
    val["predicted_probability"] = model.predict_proba(
        X_val
    )[:, 1]

    # Join day-90 revenue for business evaluation only.
    val = val.merge(
        revenue_pdf,
        on="third_party_id",
        how="left"
    )

    val["observable_revenue_90d"] = (
        val["observable_revenue_90d"].fillna(0)
    )

    # Select the 20 highest-scoring validation users.
    top20 = val.nlargest(20, "predicted_probability")

    precision_at_20 = top20["target"].sum() / 20

    total_positives = val["target"].sum()

    recall_at_20 = (
        top20["target"].sum() / total_positives
        if total_positives > 0 else 0
    )

    prevalence = val["target"].mean()

    lift_at_20 = (
        precision_at_20 / prevalence
        if prevalence > 0 else 0
    )

    total_revenue = val["observable_revenue_90d"].sum()

    revenue_captured_at_20 = (
        top20["observable_revenue_90d"].sum() / total_revenue
        if total_revenue > 0 else 0
    )

    results = {
        "model": model_name,
        "train_users": len(train),
        "validation_users": len(val),
        "train_positive_users": int(y_train.sum()),
        "validation_positive_users": int(y_val.sum()),
        "ROC_AUC": roc_auc_score(
            y_val, val["predicted_probability"]
        ),
        "PR_AUC": average_precision_score(
            y_val, val["predicted_probability"]
        ),
        "Precision_at_20": precision_at_20,
        "Recall_at_20": recall_at_20,
        "Lift_at_20": lift_at_20,
        "Revenue_Captured_at_20": revenue_captured_at_20
    }

    return model, results


# Train Model A and Model B.
model_a_lr, results_a = train_logistic(
    "default.groww_model_a_split",
    "Model A - Behaviour Only"
)

model_b_lr, results_b = train_logistic(
    "default.groww_model_b_split",
    "Model B - Behaviour + Revenue"
)

comparison = pd.DataFrame([results_a, results_b])

display(comparison)


Metric	- What it tells us

ROC-AUC	- How well the model separates positive and negative users       across thresholds

PR-AUC -	How well it identifies the rare high-value class
Precision@20 -	How many of the top 20 predictions are actually high-value

Recall@20 -	What fraction of all validation high-value users are in the top 20

Lift@20	-  How much better the top 20 are than selecting users randomly

Revenue Captured@20 - 	What fraction of validation revenue comes from those 20 users

----------------------------------------------------------------------
Model A is already performing very well.

Model B achieves perfect ROC-AUC and PR-AUC, meaning it separates every positive validation user from every negative validation user. That's possible, but unusually strong for real-world user behaviour data.

My concern: Model B uses first-seven-day revenue to predict a target defined using 90-day revenue. That's legitimate predictive information, but we must verify that none of the features accidentally contains future information or the target itself.

In [0]:
%sql
SELECT
    target,
    COUNT(*) AS users,

    ROUND(AVG(
        COALESCE(stocks_revenue_7d, 0) +
        COALESCE(fno_revenue_7d, 0) +
        COALESCE(mf_revenue_7d, 0) +
        COALESCE(ipo_revenue_7d, 0)
    ), 2) AS avg_observable_revenue_7d,

    ROUND(AVG(stocks_revenue_7d), 2)
        AS avg_stocks_revenue_7d,

    ROUND(AVG(fno_revenue_7d), 2)
        AS avg_fno_revenue_7d,

    ROUND(AVG(mf_revenue_7d), 2)
        AS avg_mf_revenue_7d,

    ROUND(AVG(ipo_revenue_7d), 2)
        AS avg_ipo_revenue_7d

FROM default.groww_model_b_split

WHERE dataset_split = 'VALIDATION'

GROUP BY target
ORDER BY target;

In [0]:
%sql
WITH revenue AS (
    SELECT
        target,
        COALESCE(stocks_revenue_7d, 0) +
        COALESCE(fno_revenue_7d, 0) +
        COALESCE(mf_revenue_7d, 0) +
        COALESCE(ipo_revenue_7d, 0) AS revenue_7d
    FROM default.groww_model_b_split
    WHERE dataset_split = 'VALIDATION'
)

SELECT
    target,
    COUNT(*) AS users,

    SUM(CASE WHEN revenue_7d > 0
             THEN 1 ELSE 0 END) AS users_with_early_revenue,

    SUM(CASE WHEN revenue_7d = 0
             THEN 1 ELSE 0 END) AS users_without_early_revenue,

    ROUND(
        100.0 * SUM(CASE WHEN revenue_7d > 0
                         THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_with_early_revenue

FROM revenue
GROUP BY target
ORDER BY target;

In [0]:
import pandas as pd
import numpy as np

# Load Model B validation data
pdf = (
    spark.table("default.groww_model_b_split")
    .filter("dataset_split = 'VALIDATION'")
    .toPandas()
)

excluded = {
    "third_party_id",
    "signup_ts",
    "target",
    "dataset_split"
}

feature_cols = [
    c for c in pdf.columns if c not in excluded
]

# Convert numeric fields consistently
numeric_types = (
    "int", "bigint", "double", "float",
    "decimal", "smallint", "tinyint"
)

numeric_cols = [
    field.name
    for field in spark.table(
        "default.groww_model_b_split"
    ).schema.fields
    if field.name in feature_cols
    and field.dataType.simpleString().startswith(numeric_types)
]

for col in numeric_cols:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

# Generate predictions
pdf["predicted_probability"] = model_b_lr.predict_proba(
    pdf[feature_cols]
)[:, 1]

# Compare score distributions
print(
    pdf.groupby("target")["predicted_probability"]
       .agg(["count", "min", "max", "mean", "median"])
)

# Check whether every positive is ranked above every negative
max_negative_score = pdf.loc[
    pdf.target == 0, "predicted_probability"
].max()

min_positive_score = pdf.loc[
    pdf.target == 1, "predicted_probability"
].min()

print("Maximum negative score:", max_negative_score)
print("Minimum positive score:", min_positive_score)

print(
    "Perfect separation:",
    min_positive_score > max_negative_score
)

In [0]:
model_b_columns = spark.table(
    "default.groww_model_b_split"
).columns

suspicious_columns = [
    c for c in model_b_columns
    if any(term in c.lower() for term in [
        "90d", "high_value", "quintile"
    ])
]

print("Potential outcome-related columns:", suspicious_columns)

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_c_split AS

SELECT * EXCEPT (
    stocks_turnover_7d,
    stocks_buy_turnover_7d,
    stocks_sell_turnover_7d,
    mf_purchase_value_7d,
    mf_redemption_value_7d
)

FROM default.groww_model_b_split;

In [0]:
model_c_lr, results_c = train_logistic(
    "default.groww_model_c_split",
    "Model C - Behaviour + Early Revenue"
)

comparison = pd.DataFrame([
    results_a,
    results_b,
    results_c
])

display(comparison)

In [0]:

import pandas as pd
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Revenue is used only for business evaluation.
revenue_pdf = (
    spark.table("default.groww_modeling_dataset")
    .select("third_party_id", "observable_revenue_90d")
    .toPandas()
)

def evaluate_test(model, table_name, model_name):

    source = spark.table(table_name)

    excluded = {
        "third_party_id",
        "signup_ts",
        "target",
        "dataset_split"
    }

    feature_cols = [
        c for c in source.columns if c not in excluded
    ]

    numeric_types = (
        "int", "bigint", "double", "float",
        "decimal", "smallint", "tinyint"
    )

    numeric_cols = [
        field.name
        for field in source.schema.fields
        if field.name in feature_cols
        and field.dataType.simpleString().startswith(numeric_types)
    ]

    test = (
        source
        .filter("dataset_split = 'TEST'")
        .select(
            "third_party_id",
            "target",
            *feature_cols
        )
        .toPandas()
    )

    for col in numeric_cols:
        test[col] = pd.to_numeric(
            test[col], errors="coerce"
        )

    y_test = test["target"].astype(int)

    # Predict using the previously trained model.
    test["predicted_probability"] = model.predict_proba(
        test[feature_cols]
    )[:, 1]

    # Add revenue only for business evaluation.
    test = test.merge(
        revenue_pdf,
        on="third_party_id",
        how="left"
    )

    test["observable_revenue_90d"] = (
        test["observable_revenue_90d"].fillna(0)
    )

    # Rank all test users and select the top 20.
    top20 = test.nlargest(20, "predicted_probability")

    selected_high_value = int(top20["target"].sum())
    total_high_value = int(y_test.sum())

    precision = selected_high_value / 20

    recall = (
        selected_high_value / total_high_value
        if total_high_value > 0 else 0
    )

    prevalence = y_test.mean()

    lift = (
        precision / prevalence
        if prevalence > 0 else 0
    )

    total_revenue = test["observable_revenue_90d"].sum()

    revenue_captured = (
        top20["observable_revenue_90d"].sum() / total_revenue
        if total_revenue > 0 else 0
    )

    return {
        "model": model_name,
        "test_users": len(test),
        "test_high_value_users": total_high_value,
        "ROC_AUC": roc_auc_score(
            y_test, test["predicted_probability"]
        ),
        "PR_AUC": average_precision_score(
            y_test, test["predicted_probability"]
        ),
        "Precision_at_20": precision,
        "Recall_at_20": recall,
        "Lift_at_20": lift,
        "Revenue_Captured_at_20": revenue_captured
    }


# Evaluate the two shortlisted models.
test_a = evaluate_test(
    model_a_lr,
    "default.groww_model_a_split",
    "Model A - Behaviour Only"
)

test_c = evaluate_test(
    model_c_lr,
    "default.groww_model_c_split",
    "Model C - Behaviour + Early Revenue"
)

display(pd.DataFrame([test_a, test_c]))


In [0]:

import pandas as pd
import numpy as np

# Load Model C feature schema
source = spark.table("default.groww_model_c_split")

excluded = {
    "third_party_id",
    "signup_ts",
    "target",
    "dataset_split"
}

feature_cols = [
    c for c in source.columns if c not in excluded
]

# 1. Check for suspicious outcome-related features
suspicious = [
    c for c in feature_cols
    if any(term in c.lower() for term in [
        "90d", "high_value", "quintile", "day90",
        "outcome", "label"
    ])
]

print("Suspicious predictor columns:", suspicious)

# 2. Extract transformed feature names and coefficients
preprocessor = model_c_lr.named_steps["preprocessor"]
classifier = model_c_lr.named_steps["classifier"]

transformed_names = preprocessor.get_feature_names_out()

coef_df = pd.DataFrame({
    "feature": transformed_names,
    "coefficient": classifier.coef_[0]
})

coef_df["absolute_coefficient"] = (
    coef_df["coefficient"].abs()
)

print("\nTop 20 influential features:")

display(
    coef_df.sort_values(
        "absolute_coefficient",
        ascending=False
    ).head(20)
)

# 3. Verify no user appears in multiple partitions
split_check = (
    spark.table("default.groww_model_b_split")
    .groupBy("third_party_id")
    .agg(
        __import__("pyspark").sql.functions.countDistinct(
            "dataset_split"
        ).alias("split_count")
    )
    .filter("split_count > 1")
    .count()
)

print("\nUsers appearing in multiple splits:", split_check)


In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_b_features AS

SELECT
    a.*,

    -- Early revenue
    d.stocks_revenue_7d,
    d.fno_revenue_7d,
    d.mf_revenue_7d,
    d.ipo_revenue_7d,
    d.observable_revenue_7d,

    -- Transaction values
    d.stocks_turnover_7d,
    d.stocks_buy_turnover_7d,
    d.stocks_sell_turnover_7d,
    d.mf_purchase_value_7d,
    d.mf_redemption_value_7d

FROM default.groww_model_a_features a

INNER JOIN default.groww_day7_features d
    ON a.third_party_id = d.third_party_id;

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_c_features AS

SELECT
    a.*,

    d.stocks_revenue_7d,
    d.fno_revenue_7d,
    d.mf_revenue_7d,
    d.ipo_revenue_7d,
    d.observable_revenue_7d

FROM default.groww_model_a_features a

INNER JOIN default.groww_day7_features d
    ON a.third_party_id = d.third_party_id;

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_b_split AS

SELECT
    b.*,
    s.dataset_split

FROM default.groww_model_b_features b

INNER JOIN default.groww_temporal_split s
    ON b.third_party_id = s.third_party_id;

In [0]:
%sql
CREATE OR REPLACE TABLE default.groww_model_c_split AS

SELECT
    c.*,
    s.dataset_split

FROM default.groww_model_c_features c

INNER JOIN default.groww_temporal_split s
    ON c.third_party_id = s.third_party_id;

In [0]:
for table_name in [
    "default.groww_model_a_split",
    "default.groww_model_b_split",
    "default.groww_model_c_split"
]:
    columns = spark.table(table_name).columns

    features = [
        c for c in columns
        if c not in {
            "third_party_id",
            "signup_ts",
            "target",
            "dataset_split"
        }
    ]

    suspicious = [
        c for c in features
        if any(term in c.lower() for term in [
            "90d", "high_value", "quintile",
            "day90", "outcome", "label"
        ])
    ]

    print(table_name)
    print("Suspicious predictors:", suspicious)

In [0]:
model_a_lr, results_a = train_logistic(
    "default.groww_model_a_split",
    "Model A - Behaviour Only"
)

model_b_lr, results_b = train_logistic(
    "default.groww_model_b_split",
    "Model B - Behaviour + Revenue + Transaction Values"
)

model_c_lr, results_c = train_logistic(
    "default.groww_model_c_split",
    "Model C - Behaviour + Early Revenue"
)

display(pd.DataFrame([
    results_a,
    results_b,
    results_c
]))

In [0]:
# Train using corrected feature tables

model_a_lr, results_a = train_logistic(
    "default.groww_model_a_split",
    "Model A - Behaviour Only"
)

model_b_lr, results_b = train_logistic(
    "default.groww_model_b_split",
    "Model B - Behaviour + Revenue + Transaction Values"
)

model_c_lr, results_c = train_logistic(
    "default.groww_model_c_split",
    "Model C - Behaviour + Early Revenue"
)

validation_comparison = pd.DataFrame([
    results_a,
    results_b,
    results_c
])

display(validation_comparison)

In [0]:
test_a = evaluate_test(
    model_a_lr,
    "default.groww_model_a_split",
    "Model A - Behaviour Only"
)

test_b = evaluate_test(
    model_b_lr,
    "default.groww_model_b_split",
    "Model B - Behaviour + Revenue + Transaction Values"
)

test_c = evaluate_test(
    model_c_lr,
    "default.groww_model_c_split",
    "Model C - Behaviour + Early Revenue"
)

test_comparison = pd.DataFrame([
    test_a,
    test_b,
    test_c
])

display(test_comparison)